In [22]:
pip install pyampute

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import kagglehub

from scipy.stats import (
    mannwhitneyu,
    chi2_contingency,
    pointbiserialr
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [7]:
warnings.filterwarnings("ignore")
# CONFIGURATION
KAGGLE_DATASET = "mubashirsidiki/student-academic-performance-500-students"      # <-- CHANGE THIS
TARGET = "parent_education"

In [8]:
path = kagglehub.dataset_download(KAGGLE_DATASET)
print("Dataset path:", path)

csv_files = glob.glob(os.path.join(path, "*.csv"))

if len(csv_files) == 0:
    raise Exception("No CSV file found.")

print("Loading:", csv_files[0])

df = pd.read_csv(csv_files[0])

print(df.shape)
print()

Dataset path: C:\Users\Administrator\.cache\kagglehub\datasets\mubashirsidiki\student-academic-performance-500-students\versions\4
Loading: C:\Users\Administrator\.cache\kagglehub\datasets\mubashirsidiki\student-academic-performance-500-students\versions\4\student_performance.csv
(500, 11)



In [9]:
df["missing_target"] = df[TARGET].isna().astype(int)

print("Missing percentage:")
print(df["missing_target"].mean()*100)
print()

Missing percentage:
23.400000000000002



In [11]:

ignore_cols = [TARGET, "missing_target"]

numeric_cols = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in ignore_cols
]

categorical_cols = [
    c for c in df.select_dtypes(exclude=np.number).columns
    if c not in ignore_cols
]

In [12]:
numeric_results = []

for col in numeric_cols:

    observed = df.loc[df.missing_target==0, col].dropna()
    missing = df.loc[df.missing_target==1, col].dropna()

    if len(observed) < 5 or len(missing) < 5:
        continue

    u,p = mannwhitneyu(
        observed,
        missing,
        alternative="two-sided"
    )

    corr, corr_p = pointbiserialr(
        df["missing_target"],
        df[col].fillna(df[col].median())
    )

    numeric_results.append({
        "Feature":col,
        "MannWhitney_p":p,
        "PointBiserial":corr,
        "Correlation_p":corr_p
    })

numeric_results = pd.DataFrame(numeric_results)
numeric_results = numeric_results.sort_values("MannWhitney_p")

print(numeric_results)

                Feature  MannWhitney_p  PointBiserial  Correlation_p
0                   age       0.011236      -0.113487       0.011101
3        previous_score       0.167424       0.061336       0.170882
2       attendance_rate       0.481168       0.030379       0.497930
4           final_score       0.510916       0.028402       0.526321
1  study_hours_per_week       0.547585       0.025109       0.575387


In [14]:
def cramers_v(table):

    chi2 = chi2_contingency(table)[0]

    n = table.sum().sum()

    phi2 = chi2/n

    r,k = table.shape

    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))

    rcorr = r - ((r-1)**2)/(n-1)

    kcorr = k - ((k-1)**2)/(n-1)

    return np.sqrt(phi2corr/min((kcorr-1),(rcorr-1)))

In [15]:
cat_results = []

for col in categorical_cols:

    table = pd.crosstab(
        df[col].fillna("MISSING_CATEGORY"),
        df["missing_target"]
    )

    if table.shape[0] < 2:
        continue

    chi2,p,_,_ = chi2_contingency(table)

    cv = cramers_v(table)

    cat_results.append({
        "Feature":col,
        "ChiSquare_p":p,
        "CramersV":cv
    })

cat_results = pd.DataFrame(cat_results)
cat_results = cat_results.sort_values("ChiSquare_p")

print(cat_results)

           Feature  ChiSquare_p  CramersV
1           gender     0.442723       0.0
0       student_id     0.478972       NaN
3  extracurricular     0.485932       0.0
4           passed     0.671810       0.0
2  internet_access     0.989211       0.0


In [16]:
X = df.drop(columns=[TARGET, "missing_target"])

y = df["missing_target"]


preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_cols
        ),

        (
            "cat",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_cols
        )

    ]

)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)

model = Pipeline([
    ("prep", preprocessor),
    ("rf", rf)
])

X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=.2,
    random_state=42,
    stratify=y
)

model.fit(X_train,y_train)

pred = model.predict_proba(X_test)[:,1]

auc = roc_auc_score(y_test,pred)

print("ROC-AUC:",round(auc,4))

if auc < .60:
    print("Evidence is consistent with MCAR.")
elif auc < .75:
    print("Weak evidence against MCAR.")
else:
    print("Strong evidence for MAR (missingness is predictable).")

ROC-AUC: 0.5381
Evidence is consistent with MCAR.


In [17]:
perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42,
    scoring="roc_auc"
)

importance = pd.DataFrame({

    "Feature":X.columns,

    "Importance":perm.importances_mean

})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(20))

                Feature  Importance
6       extracurricular    0.005901
9                passed    0.005901
1                gender    0.003444
7        previous_score    0.000988
0            student_id    0.000000
2                   age   -0.008667
5       internet_access   -0.017081
4       attendance_rate   -0.029249
8           final_score   -0.041135
3  study_hours_per_week   -0.046810


In [18]:
encoded = pd.get_dummies(
    X,
    dummy_na=True
)

encoded = encoded.fillna(encoded.median())

mi = mutual_info_classif(
    encoded,
    y,
    random_state=42
)

mi_table = pd.DataFrame({

    "Feature":encoded.columns,

    "MutualInformation":mi

})

mi_table = mi_table.sort_values(
    "MutualInformation",
    ascending=False
)

print(mi_table.head(30))

                 Feature  MutualInformation
139   student_id_STU0135           0.044885
322   student_id_STU0318           0.044323
196   student_id_STU0192           0.043855
78    student_id_STU0074           0.042951
116   student_id_STU0112           0.042444
346   student_id_STU0342           0.042285
291   student_id_STU0287           0.042073
28    student_id_STU0024           0.041844
130   student_id_STU0126           0.040495
106   student_id_STU0102           0.039939
102   student_id_STU0098           0.039201
245   student_id_STU0241           0.039099
274   student_id_STU0270           0.039074
158   student_id_STU0154           0.038806
460   student_id_STU0456           0.038160
38    student_id_STU0034           0.037662
216   student_id_STU0212           0.036482
230   student_id_STU0226           0.035355
232   student_id_STU0228           0.034861
457   student_id_STU0453           0.034574
335   student_id_STU0331           0.033747
435   student_id_STU0431        

In [23]:
try:

    from pyampute.exploration.mcar_statistical_tests import MCARTest

    tester = MCARTest(method="little")

    p = tester.little_mcar_test(
        df.drop(columns=["missing_target"])
    )

    print("Little MCAR p-value:",p)

    if p < .05:
        print("Reject MCAR.")
    else:
        print("Cannot reject MCAR.")

except Exception as e:

    print("Little's MCAR test unavailable.")
    print(e)

Little's MCAR test unavailable.
Cannot perform reduction 'mean' with string dtype


In [26]:
print()
print("="*70)
print("SUMMARY")
print("="*70)

print(f"Rows                : {len(df)}")
print(f"Missing percentage  : {100*df['missing_target'].mean():.2f}%")
print(f"ROC-AUC             : {auc:.4f}")

print("\nTop numerical associations:")
print(numeric_results.head())

print("\nTop categorical associations:")
print(cat_results.head())

print("\nTop mutual information:")
print(mi_table.head(10))

print("\nTop important variables:")
print(importance.head(10))


SUMMARY
Rows                : 500
Missing percentage  : 23.40%
ROC-AUC             : 0.5381

Top numerical associations:
                Feature  MannWhitney_p  PointBiserial  Correlation_p
0                   age       0.011236      -0.113487       0.011101
3        previous_score       0.167424       0.061336       0.170882
2       attendance_rate       0.481168       0.030379       0.497930
4           final_score       0.510916       0.028402       0.526321
1  study_hours_per_week       0.547585       0.025109       0.575387

Top categorical associations:
           Feature  ChiSquare_p  CramersV
1           gender     0.442723       0.0
0       student_id     0.478972       NaN
3  extracurricular     0.485932       0.0
4           passed     0.671810       0.0
2  internet_access     0.989211       0.0

Top mutual information:
                Feature  MutualInformation
139  student_id_STU0135           0.044885
322  student_id_STU0318           0.044323
196  student_id_STU0192    